In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 2.3 MB/s eta 0:00:44
    --------------------------------------- 1.3/101.7 MB 2.4 MB/s eta 0:00:42
    --------------------------------------- 2.1/101.7 MB 2.7 MB/s eta 0:00:38
   - -------------------------------------- 2.6/101.7 MB 2.6 MB/s eta 0:00:38
   - -------------------------------------- 3.1/101.7 MB 2.8 MB/s eta 0:00:36
   - -------------------------------------- 4.2/101.7 MB 3.0 MB/s eta 0:00:33
   - -------------------------------------- 5.0/101.7 MB 3.1 MB/s eta 0:00:32
   -- ------------------------------------- 5.8/101.7 MB 3.3 MB/s eta 0:00:29
   -- ------------------------------------- 7.1/101.7 MB 3.5 MB/s eta 0:00:27
   --- ------------------------------------ 8.4/101.7 MB 3.8 MB/s eta 0:00:25
   --- 


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("../data/atm_transactions.csv")

In [ ]:
df = df.drop(['atmName', 'atmAddress'], axis=1, errors='ignore')
df['atmId'] = pd.factorize(df['atmId'])[0]

df['transactionTime'] = pd.to_datetime(df['transactionTime'])
df = df.sort_values(['atmId', 'transactionTime'])

In [ ]:
df['day_of_week'] = df['transactionTime'].dt.weekday
df['month'] = df['transactionTime'].dt.month
df['day'] = df['transactionTime'].dt.day
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df['lag_1'] = df.groupby('atmId')['totalOutcome'].shift(1)
df['lag_7'] = df.groupby('atmId')['totalOutcome'].shift(7)

df['rolling_7'] = df.groupby('atmId')['totalOutcome'].shift(1).rolling(7).mean()
df['rolling_3'] = df.groupby('atmId')['totalOutcome'].shift(1).rolling(3).mean()

df = df.dropna()
df = df[df['totalOutcome'] > 1000]


In [ ]:
df = pd.get_dummies(df, columns=['atmCity'], drop_first=True)

In [ ]:
y = df['totalOutcome']
X = df.drop(['totalOutcome', 'transactionTime'], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [ ]:
model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.02,
    max_depth=10,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1))) * 100

print("\n MODEL PERFORMANCE")
print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R²   :", r2)
print("MAPE :", mape, "%")

In [ ]:
results = X_test.copy()
results['Actual'] = y_test
results['Predicted'] = y_pred

In [ ]:
threshold = 40000
results['Refill_Needed'] = (results['Predicted'] < threshold).astype(int)

results.to_csv("atm_predictions_final.csv", index=False)

In [ ]:
plt.figure()
plt.plot(y_test.values[:100], label="Actual")
plt.plot(y_pred[:100], label="Predicted")
plt.legend()
plt.title("ATM Cash Demand Prediction")
plt.xlabel("Time")
plt.ylabel("Cash Withdrawal")
plt.show()

In [ ]:
atm_locations = {
    atm: (random.uniform(15, 18), random.uniform(78, 83))
    for atm in df['atmId'].unique()
}

In [ ]:
refill_atms = results[results['Refill_Needed'] == 1]['atmId'].unique()

print("\nATMs needing refill:", refill_atms)


In [ ]:
from math import radians, sin, cos, sqrt, atan2

def distance(coord1, coord2):
    R = 6371
    lat1, lon1 = coord1
    lat2, lon2 = coord2

    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

nodes = list(refill_atms)

if len(nodes) > 1:

    distance_matrix = []
    for i in nodes:
        row = []
        for j in nodes:
            row.append(distance(atm_locations[i], atm_locations[j]))
        distance_matrix.append(row)

    from ortools.constraint_solver import routing_enums_pb2
    from ortools.constraint_solver import pywrapcp

    manager = pywrapcp.RoutingIndexManager(len(nodes), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return int(distance_matrix[
            manager.IndexToNode(from_index)][manager.IndexToNode(to_index)] * 1000)

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    route = []
    index = routing.Start(0)
    while not routing.IsEnd(index):
        route.append(nodes[manager.IndexToNode(index)])
        index = solution.Value(routing.NextVar(index))
    route.append(nodes[manager.IndexToNode(index)])

    print("Optimized Route:", route)

In [ ]:
import folium

    m = folium.Map(location=[16.5, 80.5], zoom_start=6)

    for atm in nodes:
        lat, lon = atm_locations[atm]
        folium.Marker([lat, lon], popup=f"ATM {atm}").add_to(m)

    route_coords = [atm_locations[i] for i in route]
    folium.PolyLine(route_coords, color="blue").add_to(m)

    m.save("atm_route_map.html")
    print("Map saved as atm_route_map.html")

else:
    print("Not enough ATMs for route optimization")